In [1]:
import pandas as pd
import json
import os
import requests
import matplotlib.pyplot as plt
from prophet import Prophet
from datetime import datetime, timedelta

In [2]:
def get_the_access(auth_url='https://auth.fractalite.com/auth/realms/master/protocol/openid-connect/token',
                   username="ml@fractalite.com",
                   scope="offline_access",
                   grant_type="password",
                   client_id="ml-beds",
                   client_secret="b33425cc-3230-48a7-8bfa-b2b779286b8c",
                   password="fractalite"):
    
    auth_payload = {
        "client_id": client_id,
        "client_secret": client_secret,
        "username": username,
        "password": password,
        "grant_type": grant_type,
        "scope": scope
    }

    token_response = requests.post(auth_url, data=auth_payload)

    if token_response.status_code == 200:
        token_data = token_response.json()
        access_token = token_data["access_token"]
        print("Token obtained successfully!")
        return access_token
    else:
        print(f"Error {token_response.status_code}: {token_response.text}")
        return None
    
def get_the_data_api(api_url='http://beds-api.fractalite.com/properties/147/rate-bases', 
                     access_token=None):
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }
    try:
        response = requests.get(api_url, headers=headers)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as err:
        print(f"The are error in the request: {err}")
        return None
    
def get_data(api_url):
    access_token = get_the_access()
    data = get_the_data_api(access_token=access_token, api_url=api_url)
    return data

def append_to_json_file(data, file):
    if os.path.exists(file):
        return 
    
    with open(file, 'w', encoding='utf-8') as json_file:
        json.dump(data, json_file, indent=4, ensure_ascii=False)
    print(f"Data saved to {file}")

In [3]:
def load_and_transform_data_grouped(file_path):
    """
    change the file of JSON to a sutable CSV file and return the RECORDS of each date
    """
    
    with open(file_path, 'r') as f:
        data = json.load(f)

    records = []

    for date, units_data in data.items():
        row = {'date': date}
        
        for unit, values in units_data.items():
            suffix = f"_{unit}"  

            price_totals = values.get('price_totals', {})
            allotment_totals = values.get('allotment_totals', {})

            row[f"total_price{suffix}"] = price_totals.get('total_price', None)
            row[f"total_public{suffix}"] = price_totals.get('total_public', None)
            row[f"total_available{suffix}"] = allotment_totals.get('total_available', None)
            row[f"total_booked{suffix}"] = allotment_totals.get('total_booked', None)
            row[f"taux_occupation{suffix}"] = allotment_totals.get('taux_occupation', None)
            row[f"taux_remplissage{suffix}"] = allotment_totals.get('taux_remplissage', None)

        records.append(row)

    return records

In [4]:
transformed_data = load_and_transform_data_grouped('../Data/UnitsTransformedData/DataFinalTransformed.json')
append_to_json_file(transformed_data, '../Data/UnitsFinalCSVData/FinalJSONdata.json')

Data saved to ../Data/UnitsFinalCSVData/FinalJSONdata.json


In [6]:
def clean_json_data(json_file_path):
    """
    Clean JSON data by removing specific columns and converting date format
    """
    df = pd.read_json(json_file_path)
    
    columns_drop = [
        "total_price_187", "total_public_187", "total_available_187", 
        "total_booked_187", "taux_occupation_187", "taux_remplissage_187", 
        "taux_remplissage_167", "taux_remplissage_166"
    ]
    
    df.drop(columns=[col for col in columns_drop if col in df.columns], inplace=True)
    
    columns_drop_2 = [
        'total_available_167', 'total_public_167', 'total_booked_167',
        'total_available_166', 'total_public_166', 'total_booked_166'
    ]
    
    df.drop(columns=[col for col in columns_drop_2 if col in df.columns], inplace=True)
    
    df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
    
    df = df.sort_values('date').reset_index(drop=True)

    df = df.set_index('date', drop=True)
    
    return df

In [7]:
df = clean_json_data("../Data/UnitsFinalCSVData/FinalJSONdata.json")

In [10]:
df.tail(5)

,total_price_166,taux_occupation_166,total_price_167,taux_occupation_167
date,,,,
2025-12-27,51088.450,0.0,9209.0,0.0
2025-12-28,65298.725,0.0,9209.0,0.0
2025-12-29,65298.725,0.0,9209.0,0.0
2025-12-30,65298.725,0.0,9209.0,0.0
2025-12-31,65298.725,0.0,9209.0,0.0


In [11]:
df.to_csv("../Data/UnitsFinalCSVData/FinalCSVdata.csv", index=True)